# Delegation and Parallel Crews [Step 5 - Multi-Crew Orchestration]

> **MLCourse - Agentic AI - CrewAI Flows and Orchestration**

CrewAI supports advanced orchestration patterns including agent delegation,
parallel crew execution via `kickoff_async`, and fan-out/fan-in patterns
within Flows. This module demonstrates how to build systems where multiple
crews run in parallel and their results are aggregated.

## What you will learn

1. `allow_delegation=True` for agent-to-agent delegation chains.
2. The supervisor pattern where one agent coordinates others.
3. `kickoff_async` for running crews asynchronously.
4. Fan-out/fan-in pattern: multiple parallel crews + aggregator.
5. Running multiple crews inside a single Flow.

## Key takeaways

- Delegation lets agents hand off subtasks to other agents.
- `kickoff_async` returns a coroutine for non-blocking crew execution.
- Flows orchestrate multiple crews with typed state passing results between them.
- Fan-out/fan-in is powerful for parallel research and aggregation.

In [ ]:
# --- Standard library imports -------------------------------------------------
import os                           # Environment variable access
import asyncio                      # Async support for parallel crews
from pathlib import Path            # OOP path handling

# --- Third-party imports ------------------------------------------------------
from dotenv import load_dotenv      # Load .env into os.environ

TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("Setup complete. Track root resolved to:", TRACK)

## 1 -- Verify Ollama availability

In [ ]:
from langchain_ollama import ChatOllama

try:
    _test = ChatOllama(model="llama3.1:8b", temperature=0)
    _test.invoke("ping")
    LLM_AVAILABLE = True
    print("[GREEN] Ollama reachable -- full pipeline will run")
except Exception as exc:
    LLM_AVAILABLE = False
    print("[WARN] Ollama not reachable:", exc)
    print("Delegation and parallel patterns demonstrated without LLM calls")

## 2 -- Import CrewAI orchestration classes

In [ ]:
from pydantic import BaseModel
from crewai import Flow, Agent, Task, Crew
from crewai.flow import start, listen

print("CrewAI orchestration imports successful")

## 3 -- Allow delegation: Agent-to-agent handoff

When `allow_delegation=True` is set on an Agent, that agent can delegate
subtasks to other agents in the same Crew. The delegating agent uses an
LLM to decide which agent to hand off to, based on the agent's role and
goal descriptions.

This creates a natural hierarchy where a coordinator agent distributes
work to specialist agents.

In [ ]:
llm = ChatOllama(model="llama3.1:8b", temperature=0) if LLM_AVAILABLE else None

# Supervisor agent -- coordinates the team
supervisor = Agent(
    role="Research Supervisor",
    goal="Coordinate research activities and delegate to specialists",
    backstory=(
        "You are a senior research manager. You break down complex questions "
        "and delegate to the right specialist. You always verify the quality "
        "of work before reporting results."
    ),
    llm=llm,
    allow_delegation=True,       # Can delegate to other agents in the crew
    verbose=False,
)

# Specialist agents
data_analyst = Agent(
    role="Data Analyst",
    goal="Analyze quantitative data and produce statistical summaries",
    backstory="You are an expert at finding patterns in numerical data.",
    llm=llm,
    verbose=False,
)

domain_expert = Agent(
    role="Domain Expert",
    goal="Provide deep domain knowledge and contextual insights",
    backstory="You have 15 years of experience in the field and provide expert context.",
    llm=llm,
    verbose=False,
)

writer = Agent(
    role="Report Writer",
    goal="Synthesize findings into a clear, concise report",
    backstory="You are skilled at combining multiple sources into cohesive reports.",
    llm=llm,
    verbose=False,
)

print("Delegation agents created:")
print(f"  Supervisor: allow_delegation={supervisor.allow_delegation}")
print(f"  Data Analyst: allow_delegation={data_analyst.allow_delegation}")
print(f"  Domain Expert: allow_delegation={domain_expert.allow_delegation}")
print(f"  Writer: allow_delegation={writer.allow_delegation}")

## 4 -- Delegation in action: Supervisor pattern

The supervisor agent receives the overall task and uses `allow_delegation`
to hand off subtasks. When the supervisor decides a specialist is needed,
it calls a tool that routes the work to the appropriate agent. The
specialist returns results to the supervisor, who integrates them.

In [ ]:
# Create a crew with delegation
delegation_task = Task(
    description=(
        "Provide a comprehensive analysis of {topic}. "
        "Delegate data analysis to the Data Analyst, "
        "domain context to the Domain Expert, "
        "and compile the final report yourself."
    ),
    expected_output="A comprehensive analysis combining data, domain expertise, and clear writing.",
    agent=supervisor,
)

delegation_crew = Crew(
    agents=[supervisor, data_analyst, domain_expert, writer],
    tasks=[delegation_task],
    verbose=False,
)

print("Delegation crew created with supervisor pattern")
print(f"  Agents: {len(delegation_crew.agents)}")
print(f"  Supervisor can delegate to: Data Analyst, Domain Expert, Writer")

## 5 -- Run the delegation crew

In [ ]:
if LLM_AVAILABLE:
    result = delegation_crew.kickoff(
        inputs={"topic": "The impact of vector databases on modern RAG systems"}
    )
    print(f"\nDelegation result:\n{str(result)[:500]}...")
else:
    print("Ollama not available -- showing crew structure only")
    print("The supervisor would delegate to specialists via allow_delegation=True")

## 6 -- Parallel crew execution with kickoff_async

CrewAI crews support async execution via `kickoff_async()`. This returns
a coroutine that can be awaited, allowing multiple crews to run in
parallel. This is useful for:

- Running independent research tasks simultaneously.
- Parallel content generation for different channels.
- Multi-source data gathering.

In [ ]:
# Define typed state for parallel execution
class ParallelState(BaseModel):
    """State for parallel crew execution with fan-out/fan-in."""
    topic: str = ""              # Input topic
    tech_research: str = ""      # Output from tech research crew
    market_research: str = ""    # Output from market research crew
    aggregated: str = ""         # Combined output from aggregator crew

print("Parallel state model defined:", list(ParallelState.model_fields.keys()))

## 7 -- Build parallel research crews

We create two independent research crews -- one for technical analysis
and one for market analysis. Both can run simultaneously because they
have no dependencies on each other.

In [ ]:
# Technical research crew
tech_researcher = Agent(
    role="Technical Researcher",
    goal="Investigate the technical aspects, architecture, and implementation details",
    backstory="You are a software architect who understands systems deeply.",
    llm=llm,
    verbose=False,
)

tech_task = Task(
    description="Research the technical aspects of {topic}",
    expected_output="A technical analysis covering architecture, implementation, and trade-offs.",
    agent=tech_researcher,
)

tech_crew = Crew(
    agents=[tech_researcher],
    tasks=[tech_task],
    verbose=False,
)

# Market research crew
market_researcher = Agent(
    role="Market Researcher",
    goal="Investigate market trends, adoption rates, and competitive landscape",
    backstory="You are a market analyst who tracks technology adoption curves.",
    llm=llm,
    verbose=False,
)

market_task = Task(
    description="Research the market aspects of {topic}",
    expected_output="A market analysis covering trends, adoption, and competitive landscape.",
    agent=market_researcher,
)

market_crew = Crew(
    agents=[market_researcher],
    tasks=[market_task],
    verbose=False,
)

print("Parallel crews defined:")
print(f"  Tech crew: {len(tech_crew.agents)} agent, {len(tech_crew.tasks)} task")
print(f"  Market crew: {len(market_crew.agents)} agent, {len(market_crew.tasks)} task")

## 8 -- Build the aggregator crew

The aggregator crew takes the outputs from both research crews and
produces a unified analysis. It runs after both parallel crews complete.

In [ ]:
aggregator = Agent(
    role="Research Aggregator",
    goal="Combine technical and market research into a unified analysis",
    backstory="You excel at synthesizing diverse research into actionable insights.",
    llm=llm,
    verbose=False,
)

aggregation_task = Task(
    description=(
        "Combine these two research reports into a unified analysis:\n"
        "TECHNICAL RESEARCH:\n{tech_research}\n\n"
        "MARKET RESEARCH:\n{market_research}"
    ),
    expected_output="A unified analysis that integrates technical and market perspectives.",
    agent=aggregator,
)

aggregator_crew = Crew(
    agents=[aggregator],
    tasks=[aggregation_task],
    verbose=False,
)

print("Aggregator crew defined")

## 9 -- Fan-out/fan-in Flow

The fan-out/fan-in pattern in a Flow:

1. **Fan-out**: The `@start()` method kicks off parallel crews.
2. **Parallel execution**: Both crews run simultaneously via `kickoff_async`.
3. **Fan-in**: An `@listen()` method receives both results and runs the
   aggregator crew.

This is the most powerful orchestration pattern in CrewAI -- it combines
the simplicity of Flows with the performance of parallel execution.

In [ ]:
class FanOutFanInFlow(Flow[ParallelState]):
    """Flow demonstrating fan-out/fan-in with parallel crews.

    The start method runs two crews in parallel (fan-out).
    The aggregation step combines their results (fan-in).
    """

    @start()
    def research_phase(self):
        """Fan-out: run tech and market research crews in parallel."""
        print(f"[Flow] Starting parallel research on: {self.state.topic}")

        if LLM_AVAILABLE:
            # Run both crews asynchronously
            import concurrent.futures

            def run_tech():
                return str(tech_crew.kickoff(inputs={"topic": self.state.topic}))

            def run_market():
                return str(market_crew.kickoff(inputs={"topic": self.state.topic}))

            # Execute in parallel using thread pool
            with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
                tech_future = executor.submit(run_tech)
                market_future = executor.submit(run_market)

                self.state.tech_research = tech_future.result()
                self.state.market_research = market_future.result()
        else:
            # Simulated results for when LLM is not available
            self.state.tech_research = f"Technical research on: {self.state.topic}"
            self.state.market_research = f"Market research on: {self.state.topic}"

        print(f"[Flow] Tech research: {len(self.state.tech_research)} chars")
        print(f"[Flow] Market research: {len(self.state.market_research)} chars")
        return {
            "tech_research": self.state.tech_research,
            "market_research": self.state.market_research,
        }

    @listen(research_phase)
    def aggregation_phase(self, research_results):
        """Fan-in: aggregate the parallel research results."""
        print("[Flow] Aggregating parallel research results")

        if LLM_AVAILABLE:
            crew = Crew(
                agents=[aggregator],
                tasks=[aggregation_task],
                verbose=False,
            )
            result = crew.kickoff(
                inputs={
                    "tech_research": research_results["tech_research"],
                    "market_research": research_results["market_research"],
                }
            )
            self.state.aggregated = str(result)
        else:
            self.state.aggregated = (
                f"Aggregated: {research_results['tech_research'][:100]} + "
                f"{research_results['market_research'][:100]}"
            )

        print(f"[Flow] Aggregation complete ({len(self.state.aggregated)} chars)")
        return self.state.aggregated

print("Fan-out/fan-in flow defined")

## 10 -- Run the fan-out/fan-in flow

In [ ]:
flow = FanOutFanInFlow()
result = flow.kickoff(inputs={"topic": "Retrieval-augmented generation in production"})

print("\n" + "=" * 60)
print("FAN-OUT/FAN-IN FLOW COMPLETE")
print("=" * 60)
print(f"Topic:      {flow.state.topic}")
print(f"Tech:       {flow.state.tech_research[:150]}...")
print(f"Market:     {flow.state.market_research[:150]}...")
print(f"Aggregated: {flow.state.aggregated[:150]}...")

## 11 -- Async flow execution

Flows also support async execution via `kickoff_async()`. This is useful
when you want to run the entire flow asynchronously, for example in a
web server or when coordinating multiple flows.

In [ ]:
# Demonstrate async kickoff pattern
async def run_flow_async():
    """Run the fan-out/fan-in flow asynchronously."""
    async_flow = FanOutFanInFlow()
    result = await async_flow.kickoff_async(
        inputs={"topic": "Async topic: Edge computing for AI"}
    )
    return async_flow

print("Async flow pattern defined")
print("Usage: result = await flow.kickoff_async(inputs={...})")
print("This runs the entire flow as a coroutine")

## 12 -- Three-crew orchestration pattern

A common production pattern is three crews:

1. **Research crews** (parallel): Gather information from multiple sources.
2. **Analysis crew**: Deep-dive into the collected data.
3. **Output crew**: Produce the final deliverable.

This separates concerns and lets you scale each layer independently.

In [ ]:
# Define the three-crew pipeline structure
class ThreeCrewPipeline(Flow[ParallelState]):
    """Three-crew pipeline: parallel research -> analysis -> output."""

    @start()
    def parallel_research(self):
        """Phase 1: Run multiple research crews in parallel."""
        print("[Flow] Phase 1: Parallel research")
        # In production, this would fan out to N research crews
        self.state.tech_research = f"Tech findings on {self.state.topic}"
        self.state.market_research = f"Market findings on {self.state.topic}"
        return self.state

    @listen(parallel_research)
    def analyze(self, state):
        """Phase 2: Analyze collected research."""
        print("[Flow] Phase 2: Analysis")
        self.state.aggregated = (
            f"Analysis combining tech and market research on {self.state.topic}"
        )
        return self.state.aggregated

    @listen(analyze)
    def produce_output(self, analysis):
        """Phase 3: Produce final deliverable."""
        print("[Flow] Phase 3: Output generation")
        final = f"Final deliverable based on: {analysis}"
        return final

print("Three-crew pipeline defined")

## 13 -- Summary

CrewAI provides several orchestration patterns for multi-crew systems:

| Pattern | Mechanism | Use Case |
|---------|-----------|----------|
| Delegation | `allow_delegation=True` | Agent-to-agent handoff |
| Parallel | `kickoff_async` | Independent crews run simultaneously |
| Fan-out/fan-in | Flow `@start` + `@listen` | Parallel research + aggregation |
| Three-crew | Flow with multiple steps | Research -> Analysis -> Output |

Key points:
- `allow_delegation` creates natural agent hierarchies.
- `kickoff_async` enables non-blocking parallel execution.
- Flows orchestrate multiple crews with typed state.
- Fan-out/fan-in is the most powerful pattern for parallel work.

In [ ]:
print("=" * 60)
print("MODULE SUMMARY -- Delegation and Parallel Crews")
print("=" * 60)
print()
print("Delegation:")
print("  Agent(..., allow_delegation=True)")
print("  - Agent can hand off subtasks to other agents in the crew")
print("  - Supervisor pattern: one coordinator + multiple specialists")
print()
print("Parallel execution:")
print("  crew.kickoff_async()  -- returns a coroutine")
print("  await flow.kickoff_async()  -- async flow execution")
print()
print("Fan-out/fan-in:")
print("  @start() -> runs parallel crews (fan-out)")
print("  @listen() -> aggregates results (fan-in)")
print()
print("Three-crew pipeline:")
print("  Phase 1: Parallel research crews")
print("  Phase 2: Analysis crew")
print("  Phase 3: Output/delivery crew")